# GW170817 Parameter Estimation with `bilby` + `mlgw_bns_jax`

This notebook performs Bayesian parameter estimation on the GW170817 event using:
- **`mlgw_bns_jax`**: JAX-based BNS waveform approximant
- **`bilby`**: Bayesian inference library for gravitational-wave astronomy
- **Cleaned (deglitched) GWOSC data**: version 2 strain data with the L1 glitch removed

The analysis uses the `dynesty` nested sampler.

In [ ]:
"""Imports and JAX configuration."""

from __future__ import annotations

import os
import sys

import numpy as np

# JAX configuration (must come before any JAX import)
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import bilby
import logging

logging.getLogger("bilby").setLevel(logging.INFO)
# Suppress repetitive zenith/azimuth conversion warnings
logging.getLogger("bilby").addFilter(
    lambda record: "zenith/azimuth" not in record.getMessage()
)
from bilby.gw.conversion import (
    generate_all_bns_parameters,
    convert_to_lal_binary_neutron_star_parameters,
)

print("JAX devices:", jax.devices())
print("bilby version:", bilby.__version__)

## Load the JAX waveform model

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_predict_fn = jax.jit(load_predict(MODEL_PATH))

print("Model loaded successfully.")

## Event parameters and source model

In [ ]:
# GW170817 event parameters
TRIGGER_TIME = 1187008882.43          # GPS trigger time
DURATION = 32                         # seconds of data to analyse
SAMPLING_FREQUENCY = 4096             # Hz
MAXIMUM_FREQUENCY = 2000              # Hz
MINIMUM_FREQUENCY = 20.0              # Hz  (low-frequency cut-off)
POST_TRIGGER_DURATION = 2             # seconds after trigger to keep
DATA_START_GPS = 1187008867           # GPS start of the data files
LABEL = "GW170817"
OUTDIR = "outdir_GW170817"

os.makedirs(OUTDIR, exist_ok=True)


def mlgw_bns_jax_frequency_domain_source_model(
    frequency_array: np.ndarray,
    mass_1: float,
    mass_2: float,
    luminosity_distance: float,
    theta_jn: float,
    phase: float,
    chi_1: float,
    chi_2: float,
    lambda_1: float,
    lambda_2: float,
    **kwargs,
) -> dict[str, np.ndarray]:
    """Frequency-domain BNS waveform using the mlgw_bns JAX model."""
    freq_mask = frequency_array > 0
    freqs = frequency_array[freq_mask]

    m1 = max(mass_1, mass_2)
    m2 = min(mass_1, mass_2)
    mass_ratio = m1 / m2
    total_mass = m1 + m2

    params = jnp.array([mass_ratio, lambda_1, lambda_2, chi_1, chi_2])

    hp_jax, hc_jax = _predict_fn(
        params,
        jnp.array(freqs),
        jnp.array(total_mass),
        jnp.array(luminosity_distance),
        jnp.array(theta_jn),
    )

    hp_out = np.array(hp_jax)
    hc_out = np.array(hc_jax)

    # Apply reference phase rotation
    phase_shift = np.exp(-2j * phase)
    hp_out *= phase_shift
    hc_out *= phase_shift

    hp_full = np.zeros(len(frequency_array), dtype=complex)
    hc_full = np.zeros(len(frequency_array), dtype=complex)
    hp_full[freq_mask] = hp_out
    hc_full[freq_mask] = hc_out

    return {"plus": hp_full, "cross": hc_full}

## Load cleaned (deglitched) strain data

We use the **GWOSC C01/v2** cleaned data, which has the L1 glitch removed.
These files were previously downloaded and saved as `H1_cleaned.txt`, `L1_cleaned.txt`, `V1_cleaned.txt`.

In [ ]:
DATA_DIR = "gw170817_data"

interferometers = bilby.gw.detector.InterferometerList(["H1", "L1", "V1"])
start_time = TRIGGER_TIME + POST_TRIGGER_DURATION - DURATION

# Load cleaned (deglitched) GWOSC v2 data for each detector
cleaned_files = {
    "H1": os.path.join(DATA_DIR, "H1_cleaned.txt"),
    "L1": os.path.join(DATA_DIR, "L1_cleaned.txt"),
    "V1": os.path.join(DATA_DIR, "V1_cleaned.txt"),
}

for ifo in interferometers:
    filepath = cleaned_files[ifo.name]
    print(f"Loading cleaned data for {ifo.name} from {filepath}")
    strain = np.loadtxt(filepath, comments="#")
    ifo.set_strain_data_from_frequency_domain_strain(
        sampling_frequency=SAMPLING_FREQUENCY,
        duration=DURATION,
        start_time=DATA_START_GPS,
        frequency_domain_strain=np.fft.rfft(strain) / SAMPLING_FREQUENCY,
    )
    ifo.minimum_frequency = MINIMUM_FREQUENCY
    ifo.maximum_frequency = MAXIMUM_FREQUENCY

print(f"\nLoaded {len(interferometers)} detectors with cleaned data.")

## Waveform generator and priors

In [ ]:
waveform_generator = bilby.gw.WaveformGenerator(
    duration=DURATION,
    sampling_frequency=SAMPLING_FREQUENCY,
    frequency_domain_source_model=mlgw_bns_jax_frequency_domain_source_model,
    parameter_conversion=convert_to_lal_binary_neutron_star_parameters,
    waveform_arguments=dict(minimum_frequency=MINIMUM_FREQUENCY),
)

# Define priors
priors = bilby.core.prior.PriorDict()

priors["chirp_mass"] = bilby.core.prior.Uniform(
    minimum=1.18, maximum=1.21, name="chirp_mass",
    latex_label=r"$\mathcal{M}$", unit=r"$M_\odot$",
)
priors["mass_ratio"] = bilby.core.prior.Uniform(
    minimum=0.5, maximum=1.0, name="mass_ratio",
    latex_label=r"$q$",
)
priors["chi_1"] = bilby.core.prior.Uniform(
    minimum=-0.05, maximum=0.05, name="chi_1",
    latex_label=r"$\chi_1$",
)
priors["chi_2"] = bilby.core.prior.Uniform(
    minimum=-0.05, maximum=0.05, name="chi_2",
    latex_label=r"$\chi_2$",
)
priors["lambda_1"] = bilby.core.prior.Uniform(
    minimum=0, maximum=5000, name="lambda_1",
    latex_label=r"$\Lambda_1$",
)
priors["lambda_2"] = bilby.core.prior.Uniform(
    minimum=0, maximum=5000, name="lambda_2",
    latex_label=r"$\Lambda_2$",
)
priors["luminosity_distance"] = bilby.gw.prior.UniformSourceFrame(
    minimum=10, maximum=100, name="luminosity_distance",
    latex_label=r"$d_L$", unit="Mpc",
)
priors["theta_jn"] = bilby.core.prior.Sine(
    minimum=0, maximum=np.pi, name="theta_jn",
    latex_label=r"$\theta_{JN}$",
)
priors["ra"] = bilby.core.prior.Uniform(
    minimum=0, maximum=2 * np.pi, name="ra",
    latex_label=r"$\alpha$", boundary="periodic",
)
priors["dec"] = bilby.core.prior.Cosine(
    minimum=-np.pi / 2, maximum=np.pi / 2, name="dec",
    latex_label=r"$\delta$",
)
priors["psi"] = bilby.core.prior.Uniform(
    minimum=0, maximum=np.pi, name="psi",
    latex_label=r"$\psi$", boundary="periodic",
)
priors["phase"] = bilby.core.prior.Uniform(
    minimum=0, maximum=2 * np.pi, name="phase",
    latex_label=r"$\phi$", boundary="periodic",
)
priors["geocent_time"] = bilby.core.prior.Uniform(
    minimum=TRIGGER_TIME - 0.1,
    maximum=TRIGGER_TIME + 0.1,
    name="geocent_time",
    latex_label=r"$t_c$",
    unit="s",
)

print(f"Prior parameters: {list(priors.keys())}")

## Likelihood and sampler

Construct the `GravitationalWaveTransient` likelihood with time, distance, and phase marginalisation, then run `dynesty`.

In [ ]:
bilby.core.utils.setup_logger(outdir=OUTDIR, label=LABEL, log_level="info")

likelihood = bilby.gw.GravitationalWaveTransient(
    interferometers=interferometers,
    waveform_generator=waveform_generator,
    priors=priors,
    time_marginalization=True,
    distance_marginalization=True,
    phase_marginalization=True,
    reference_frame="H1L1V1",
    jitter_time=True,
)

result = bilby.run_sampler(
    likelihood=likelihood,
    priors=priors,
    sampler="dynesty",
    npoints=1024,
    walks=100,
    nact=10,
    maxmcmc=5000,
    injection_parameters=None,
    outdir=OUTDIR,
    label=LABEL,
    conversion_function=generate_all_bns_parameters,
    result_class=bilby.gw.result.CBCResult,
)

## Results — corner plot

In [ ]:
result.plot_corner()